# Example · School places dashboard (Bokeh)

**Session 8 — Space and Time (instructor demo)** · about 20 minutes, live

A complete interactive dashboard on open data about school places in Catalonia. It shows what Bokeh
can do when Python runs behind the page. You are **not** expected to build one this size, but every
piece of it can be reused in your project.

**You will see how to**
- tidy an open dataset into the few columns a dashboard needs, once, before any plotting
- let several views share one `ColumnDataSource`, so a click on a school selects it everywhere
- write one `update()` callback that re-filters the data whenever a widget changes
- test a dashboard without opening a browser

**Data:** pre-enrolment allocation of school places, Transparència Catalunya
[`99md-r3rq`](https://analisi.transparenciacatalunya.cat/d/99md-r3rq), school years 2017/18 to
2025/26 (snapshot of 2026-10-06 in `data/education/`). One row = one school × course level × school year.

**Outline**
1. Setup
2. Load and tidy the data
3. A first look (no dashboard yet)
4. Building blocks: filter and summarise
5. The views
6. The app
7. Test it without a browser
8. Run it
9. Ideas to take it further

> **Before you run the last cell.** The dashboard is a Bokeh *server* app running inside this kernel. It
> works in JupyterLab or Jupyter Notebook (not in VS Code or a GitHub preview). Set `NOTEBOOK_URL` in the
> setup cell to the address in your browser bar, e.g. `"localhost:8889"` if Jupyter runs on port 8889.

## 1. Setup

In [ ]:
import io
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests

DATA = Path("../../data")
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

# Address of your Jupyter server as shown in the browser bar (only the last cell uses it)
NOTEBOOK_URL = "localhost:8888"

## 2. Load and tidy the data

`SOURCE = "snapshot"` reads the copy in `data/` (works offline and in CI). `SOURCE = "api"` downloads the
current version with the pagination loop of notebook 03. From 2024/25 on, the school years are written
`2024-2025` instead of `2023/2024`: a good reason not to parse a column by splitting it on one
particular character.

In [ ]:
SOURCE = "snapshot"  # or "api"


def fetch_socrata_csv(dataset_id, page_size=50_000):
    """Download every row of a Transparència Catalunya dataset as CSV, page by page."""
    url = f"https://analisi.transparenciacatalunya.cat/resource/{dataset_id}.csv"
    frames, offset = [], 0
    while True:
        params = {"$limit": page_size, "$offset": offset, "$order": ":id"}
        r = requests.get(url, params=params, timeout=300)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text)))
        offset += page_size
        if len(frames[-1]) < page_size:
            return pd.concat(frames, ignore_index=True)


if SOURCE == "api":
    raw = fetch_socrata_csv("99md-r3rq")
    DATA_NOTE = "Transparència Catalunya, dataset 99md-r3rq (live API)"
else:
    raw = pd.read_parquet(DATA / "education/school_places.parquet")
    DATA_NOTE = "Transparència Catalunya, dataset 99md-r3rq (snapshot of 2026-10-06)"
print(f"{len(raw):,} rows × {raw.shape[1]} columns")

The dashboard needs 14 of the 30 columns. `tidy()` keeps those, gives them English names and computes,
**once**, everything the callbacks will need later:

- `course`: one label per stage and level (`P3` … `ESO 4`), so the app needs one dropdown instead of
  two that can disagree (there is no "Pre-school, level 6");
- the share of the offered places filled by first-choice applicants, by other applicants, and in total.
  It is `NaN` when a school offered no places: 0 filled out of 0 is not 0 %;
- district names rebuilt from the district codes. In 2019/20 the districts of Sabadell, Terrassa and
  Santa Coloma carry *Barcelona's* names (Eixample, Gràcia…), and before that they have none. The codes
  are right, so each municipality and code gets its latest name;
- a `label` for the search box. School names repeat (15 schools are called *Escola Pompeu Fabra*), so
  the label adds the municipality and the official 8-digit school code.

A lesson from the previous snapshot of this dataset (2024): it had 150 rows twice and was missing 150
others. It had been downloaded page by page **without `$order`**, so consecutive pages overlapped. This
one was downloaded with `"$order": ":id"` (notebook 03, `scripts/download_data.py`), and the checks
below make sure every school appears only once per course and school year.

In [ ]:
COURSE_LABELS = (
    {("Educació infantil: Parvulari", n): f"P{n + 2}" for n in (1, 2, 3)}
    | {("Educació primària", n): f"Primary {n}" for n in range(1, 7)}
    | {("Educació secundària obligatòria", n): f"ESO {n}" for n in range(1, 5)}
)
COURSES = list(COURSE_LABELS.values())  # in school order: P3, P4, P5, Primary 1 … ESO 4

COLUMNS = {
    "curs": "school_year",
    "codi_centre": "school_id",
    "denominaci_completa": "school",
    "nom_naturalesa": "ownership",
    "nom_comarca": "comarca",
    "nom_municipi": "municipality",
    "codi_districte_municipal": "district_code",
    "nom_dm": "district",
    "coordenades_geo_x": "lon",
    "coordenades_geo_y": "lat",
    "places_ofertades_a_la": "offered",
    "assignacions": "assigned",
    "assignacions_1a_peticio": "first_choice",
    "assignacions_altres_peticions": "other_choice",
}


def tidy(raw):
    df = raw.copy()
    df["course"] = [
        COURSE_LABELS.get(key)
        for key in zip(df["nom_ensenyament"], df["nivell"].astype(int), strict=True)
    ]
    df = df.rename(columns=COLUMNS)[["course", *COLUMNS.values()]].dropna(subset=["course"])

    df["school_year"] = df["school_year"].str.replace("-", "/")  # "2024-2025" -> "2024/2025"
    df["year"] = df["school_year"].str[:4].astype(int)  # first calendar year of the course
    df["ownership"] = df["ownership"].map({"Públic": "Public", "Privat": "Private"})

    # District names are wrong in some years, the codes are not: use each code's latest name
    named = df.dropna(subset=["district"]).sort_values("year")
    latest_name = named.groupby(["municipality", "district_code"])["district"].last()
    df = df.drop(columns="district").join(latest_name, on=["municipality", "district_code"])

    code = df["school_id"].astype(str).str.zfill(8)  # official 8-digit school code
    df["label"] = df["school"] + " · " + df["municipality"] + " · " + code

    offered = df["offered"].where(df["offered"] > 0)  # NaN where no places were offered
    df["first_pct"] = 100 * df["first_choice"] / offered
    df["other_pct"] = 100 * df["other_choice"] / offered
    df["occupancy_pct"] = 100 * df["assigned"] / offered
    return df.reset_index(drop=True)


places = tidy(raw)
places.head(3)

### Check before you plot

A dashboard makes wrong numbers look convincing. Three assertions catch the problems this dataset could
have:

In [ ]:
assert places.eval("assigned == first_choice + other_choice").all(), "assigned ≠ 1st + other"
assert places["ownership"].notna().all(), "an ownership value other than Públic/Privat"
assert not places.duplicated(["school_year", "school_id", "course"]).any(), "a school appears twice"

SCHOOL_YEARS = sorted(places["school_year"].unique())
LATEST = SCHOOL_YEARS[-1]
YEARS = sorted(places["year"].unique())
YEAR_LABELS = {y: f"{y}/{(y + 1) % 100:02d}" for y in YEARS}  # 2023 -> "2023/24"
n_schools = places["school_id"].nunique()
print(f"{len(places):,} rows · {n_schools:,} schools · {SCHOOL_YEARS[0]} to {LATEST}")

### Where are the schools?

A map adds one more thing to check: **are the schools where the data says they are?** Each point should
fall inside the school's own municipality. The ICGC boundaries in `data/icgc/` are simplified
(1:1,000,000), so we allow 2 km of tolerance, and we measure in metres in EPSG:25831 (notebook 08).
Coordinates can change from one year to the next, so we check every school year.

In [ ]:
BOUNDARIES = DATA / "icgc/divisions-administratives-v2r1-municipis-1000000-20250730.json"
municipalities = gpd.read_file(BOUNDARIES).to_crs("EPSG:25831")
boundary_of = municipalities.set_index(municipalities["NOMMUNI"].str.lower())["geometry"]

coords = places[["school_id", "school_year", "year", "municipality", "lon", "lat"]]
coords = coords.dropna().drop_duplicates()
# A few municipality names are spelt differently in the two sources: those are not checked
known = coords["municipality"].str.lower().isin(boundary_of.index)
points = gpd.GeoSeries.from_xy(coords.loc[known, "lon"], coords.loc[known, "lat"], crs="EPSG:4326")
own_boundary = boundary_of.reindex(coords.loc[known, "municipality"].str.lower())
metres = points.to_crs("EPSG:25831").distance(own_boundary, align=False)
coords.loc[known, "km_outside"] = metres.to_numpy() / 1000

(
    coords.assign(misplaced=coords["km_outside"] > 2)
    .groupby("school_year")["misplaced"]
    .mean()
    .to_frame("points > 2 km outside their municipality")
    .style.format("{:.1%}")
)

Something went wrong in **2020/21**: almost three quarters of the points are in the wrong
municipality, as if the coordinates had been shuffled between schools. In 2021/22 the same school can
even get a different point for each course. Plotting every row at its own coordinates would scatter schools all over Catalonia.

So each school gets **one** location: its most recent coordinates that pass the check. Schools with no
point that passes are left off the map (they stay in the scatter and the table). On a real project you
would also report the problem to the publisher: that is how open data gets better.

In [ ]:
# The latest coordinates of each school that pass the check (unchecked names count as passing)
reliable = coords[~(coords["km_outside"] > 2)].sort_values("year")
location = reliable.groupby("school_id")[["lon", "lat"]].last()
mercator = gpd.GeoSeries.from_xy(location["lon"], location["lat"], crs="EPSG:4326")
mercator = mercator.to_crs("EPSG:3857")  # Web Mercator, the CRS of map tiles
location = pd.DataFrame({"map_x": mercator.x, "map_y": mercator.y})

places = places.drop(columns=["map_x", "map_y"], errors="ignore").join(location, on="school_id")
unplaced = places.loc[places["map_x"].isna(), ["school_id", "school", "municipality"]]
n_unplaced = unplaced["school_id"].nunique()
print(f"{len(location):,} schools on the map; {n_unplaced} without a reliable location:")
unplaced.drop_duplicates()

## 3. A first look (no dashboard yet)

A dashboard is for exploring a question you already understand. Two quick tables tell us where to start.

**Where do families actually apply?** Children who stay in the same school keep their place, so not every
level offers places:

In [ ]:
(
    places.groupby("course")
    .agg(
        schools=("school_id", "nunique"),
        offering_places=("offered", lambda s: (s > 0).mean()),
        places_offered=("offered", "sum"),
    )
    .reindex(COURSES)
    .style.format({"schools": "{:,}", "offering_places": "{:.0%}", "places_offered": "{:,}"})
)

P3 (age 3) and ESO 1 (age 12) are the two entry points: almost every school offers places there. At the
other levels about half of the schools offer none, and the few places offered are vacancies left over,
so their percentages say little. The dashboard therefore opens on **P3**.

**How has the P3 offer changed?**

In [ ]:
p3_offer = places.query("course == 'P3'").pivot_table(
    index="school_year", columns="ownership", values="offered", aggfunc="sum"
)
p3_offer["public share"] = p3_offer["Public"] / (p3_offer["Public"] + p3_offer["Private"])
p3_offer.style.format({"Public": "{:,}", "Private": "{:,}", "public share": "{:.0%}"})

In this snapshot the P3 places offered fall by almost a quarter between 2017/18 and 2025/26 (−23 %):
fewer children are being born. The public share stays at about two thirds, but since 2023/24 only public
schools keep cutting places, while the private offer stays flat. The dashboard lets you see *where* that
happened and which schools still fill up.

## 4. Building blocks: filter and summarise

The app calls these plain functions every time a widget changes. They take a DataFrame and return a
DataFrame, so we can test them right here, without any dashboard.

In [ ]:
ALL = "All"


def filter_places(df, course, **area):
    """Rows of one course level (every school year) in an area, e.g. municipality="Girona"."""
    mask = df["course"] == course
    for column_name, value in area.items():
        if value != ALL:
            mask &= df[column_name] == value
    return df[mask]


def summarise(df):
    """Schools, places offered and fill rates, for public, private and all schools."""
    rows = {}
    for name in ["Public", "Private", ALL]:
        g = df if name == ALL else df[df["ownership"] == name]
        offered = g["offered"].sum()
        rows[name] = {
            "schools": g.loc[g["offered"] > 0, "school_id"].nunique(),
            "places offered": offered,
            "% assigned": g["assigned"].sum() / offered if offered else np.nan,
            "% to 1st choice": g["first_choice"].sum() / offered if offered else np.nan,
        }
    return pd.DataFrame.from_dict(rows, orient="index")


def summary_table(df):
    formats = {
        "schools": "{:,}",
        "places offered": "{:,}",
        "% assigned": "{:.0%}",
        "% to 1st choice": "{:.0%}",
    }
    return summarise(df).style.format(formats, na_rep="–")


girona = filter_places(places, "P3", municipality="Girona")
summary_table(girona[girona["school_year"] == LATEST])

## 5. The views

Each view is an ordinary Bokeh figure that reads its columns from a `ColumnDataSource`. The figures are
built once; afterwards the app only replaces `source.data`, and Bokeh redraws what changed.

| View | Source: one row per … | Question it answers |
|---|---|---|
| Scatter | `schools`: school, in the selected year | Who fills the places: first-choice applicants or the rest? |
| Map | `schools` (the same source) | Where are those schools? |
| Table | `schools` (the same source) | The exact numbers, sortable |
| History | `history`: school year, for one school | How has demand for this school changed? |
| Trend | `trend`: school year, for the whole area | Public and private places offered over time |

The scatter, the map and the table share **one** source, so they share its selection too: click a school
on the map and it lights up in the scatter and the table. This is *linked brushing*, and it costs no code.

Colours keep one meaning in every view: **blue/orange = public/private** (a pair that stays distinct for
colour-blind readers), **violet/green = places assigned to first-choice / other applicants**. Each colour
is defined once, below, and the legends are generated from the data, never typed by hand.

In [ ]:
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
from bokeh.models import (
    AutocompleteInput,
    BoxAnnotation,
    Button,
    ColumnDataSource,
    DataTable,
    Div,
    HoverTool,
    Label,
    NumberFormatter,
    NumeralTickFormatter,
    Select,
    TableColumn,
)
from bokeh.plotting import figure
from bokeh.transform import factor_cmap, jitter

output_notebook(hide_banner=True)

OWNERSHIP = ["Public", "Private"]
OWNERSHIP_COLORS = ["#2a78d6", "#eb6834"]  # blue, orange
FIRST_COLOR, OTHER_COLOR, OFFERED_COLOR = "#4a3aa7", "#008300", "#52514e"
GRID_COLOR, AXIS_COLOR = "#e1e0d9", "#c3c2b7"

SCHOOL_COLUMNS = [
    "school_id",
    "label",
    "ownership",
    "offered",
    "assigned",
    "first_choice",
    "other_choice",
    "first_pct",
    "other_pct",
    "occupancy_pct",
    "map_x",
    "map_y",
]
SCHOOL_TOOLTIPS = [
    ("School", "@label"),
    ("Ownership", "@ownership"),
    ("Places offered", "@offered"),
    ("Assigned", "@assigned (@occupancy_pct{0}%)"),
    ("to 1st choice", "@first_choice (@first_pct{0}%)"),
    ("to other choices", "@other_choice (@other_pct{0}%)"),
]


def school_data(df):
    """Columns for the `schools` source: one row per school."""
    return {c: df[c].to_numpy() for c in SCHOOL_COLUMNS}


def quiet_chrome(p):
    """Thin, light grid and axes, so the data is the loudest thing in the figure."""
    p.grid.grid_line_color = GRID_COLOR
    p.axis.axis_line_color = p.axis.major_tick_line_color = AXIS_COLOR
    p.axis.minor_tick_line_color = None
    p.outline_line_color = None
    p.toolbar.logo = None
    return p

**Scatter.** Each dot is a school. Moving right, more of its places went to families who put it first;
moving up, more went to families who had chosen it lower down. On the grey diagonal all places offered were
assigned (100 % occupancy); dots below it kept vacancies. Dots are *jittered* by up to ±1 point so that
schools with identical values do not hide each other; the tooltip shows the exact values.

In [ ]:
def make_scatter(source):
    p = figure(
        title="Who fills the places? One dot per school",
        height=430,
        sizing_mode="stretch_width",
        x_axis_label="Assigned to 1st-choice applicants (% of places offered)",
        y_axis_label="Assigned to other applicants (%)",
        tools="tap,pan,wheel_zoom,box_zoom,reset",
    )
    p.line([0, 100], [100, 0], color=AXIS_COLOR, line_width=2)
    p.add_layout(
        Label(
            x=0,
            y=100,
            text=" all places assigned",
            text_font_size="11px",
            text_color=OFFERED_COLOR,
            text_baseline="bottom",
        )
    )
    dots = p.scatter(
        x=jitter("first_pct", 2),
        y=jitter("other_pct", 2),
        source=source,
        size=8,
        color=factor_cmap("ownership", OWNERSHIP_COLORS, OWNERSHIP),
        fill_alpha=0.7,
        line_color="white",
        legend_field="ownership",
        selection_line_color="black",
        selection_line_width=2,
        nonselection_fill_alpha=0.25,
        nonselection_line_alpha=0,
    )
    p.add_tools(HoverTool(renderers=[dots], tooltips=SCHOOL_TOOLTIPS))
    p.legend.location = "top_right"
    return quiet_chrome(p)


def make_map(source):
    p = figure(
        title="Where are they?",
        height=430,
        sizing_mode="stretch_width",
        match_aspect=True,
        x_axis_type="mercator",
        y_axis_type="mercator",
        tools="tap,pan,wheel_zoom,box_zoom,reset",
    )
    p.add_tile("Esri.WorldGrayCanvas")
    dots = p.scatter(
        "map_x",
        "map_y",
        source=source,
        size=7,
        color=factor_cmap("ownership", OWNERSHIP_COLORS, OWNERSHIP),
        fill_alpha=0.8,
        line_color="white",
        legend_field="ownership",
        selection_line_color="black",
        selection_line_width=2,
        nonselection_fill_alpha=0.25,
        nonselection_line_alpha=0,
    )
    p.add_tools(HoverTool(renderers=[dots], tooltips=SCHOOL_TOOLTIPS))
    p.legend.location = "bottom_right"  # the sea: no schools there
    p.axis.visible = False
    return quiet_chrome(p)


def make_table(source):
    pct = NumberFormatter(format="0")
    columns = [
        TableColumn(field="label", title="School", width=360),
        TableColumn(field="ownership", title="Ownership", width=80),
        TableColumn(field="offered", title="Places offered", width=90),
        TableColumn(field="assigned", title="Assigned", width=80),
        TableColumn(field="occupancy_pct", title="Assigned (%)", formatter=pct, width=90),
        TableColumn(field="first_pct", title="to 1st choice (%)", formatter=pct, width=110),
    ]
    return DataTable(
        source=source, columns=columns, height=250, sizing_mode="stretch_width", index_position=None
    )

**History and trend.** Both have the school year on the x-axis, show *every* year, and shade the year
picked in the filters, so the snapshot (scatter, map, table) and the trend are always read together. Their
tooltips list every series for the year under the pointer.

In [ ]:
def history_data(df):
    """Columns for the `history` source: one row per school year, for one school."""
    df = df.sort_values("year")
    return {
        "year": df["year"].to_numpy(),
        "school_year": df["year"].map(YEAR_LABELS).to_numpy(),
        **{
            c: df[c].to_numpy()
            for c in ["offered", "assigned", "first_choice", "other_choice", "occupancy_pct"]
        },
    }


def trend_data(df):
    """Columns for the `trend` source: places offered and % assigned per year and ownership."""
    data = {"year": YEARS, "school_year": [YEAR_LABELS[y] for y in YEARS]}
    for name in OWNERSHIP:
        sums = (
            df[df["ownership"] == name]
            .groupby("year")[["offered", "assigned"]]
            .sum()
            .reindex(YEARS, fill_value=0)
        )
        data[name] = sums["offered"].to_numpy()
        data[f"{name}_pct"] = (
            100 * sums["assigned"] / sums["offered"].replace(0, np.nan)
        ).to_numpy()
    return data


def years_axis(p):
    """School years on the x-axis, and a shaded band for the selected year (returned to move it)."""
    p.xaxis.ticker = YEARS
    p.xaxis.major_label_overrides = {y: YEAR_LABELS[y] for y in YEARS}
    p.xgrid.grid_line_color = None
    p.y_range.start = 0
    p.yaxis.formatter = NumeralTickFormatter(format="0,0")
    p.legend.orientation = "horizontal"
    p.legend.location = "top_left"
    p.add_layout(p.legend[0], "below")
    band = BoxAnnotation(fill_color=GRID_COLOR, fill_alpha=0.6, level="underlay")
    p.add_layout(band)
    return quiet_chrome(p), band


def make_history(source):
    p = figure(
        height=320,
        sizing_mode="stretch_width",
        toolbar_location=None,
        tools="",
        y_axis_label="Places",
    )
    p.vbar_stack(
        ["first_choice", "other_choice"],
        x="year",
        source=source,
        width=0.3,
        color=[FIRST_COLOR, OTHER_COLOR],
        line_color="white",
        line_width=2,
        legend_label=["Assigned to 1st choice", "Assigned to other choices"],
    )
    p.line(
        "year",
        "offered",
        source=source,
        color=OFFERED_COLOR,
        line_width=2,
        legend_label="Places offered",
    )
    dots = p.scatter(
        "year",
        "offered",
        source=source,
        color=OFFERED_COLOR,
        size=8,
        line_color="white",
        legend_label="Places offered",
    )
    p.add_tools(
        HoverTool(
            renderers=[dots],
            mode="vline",
            tooltips=[
                ("School year", "@school_year"),
                ("Places offered", "@offered"),
                ("Assigned", "@assigned (@occupancy_pct{0}%)"),
                ("to 1st choice", "@first_choice"),
                ("to other choices", "@other_choice"),
            ],
        )
    )
    return years_axis(p)


def make_trend(source):
    p = figure(
        height=320,
        sizing_mode="stretch_width",
        toolbar_location=None,
        tools="",
        y_axis_label="Places offered",
    )
    for name, color in zip(OWNERSHIP, OWNERSHIP_COLORS, strict=True):
        p.line("year", name, source=source, color=color, line_width=2, legend_label=name)
        dots = p.scatter(
            "year", name, source=source, color=color, size=8, line_color="white", legend_label=name
        )
    p.add_tools(
        HoverTool(
            renderers=[dots],
            mode="vline",
            tooltips=[
                ("School year", "@school_year"),
                ("Public", "@Public{0,0} places, @Public_pct{0}% assigned"),
                ("Private", "@Private{0,0} places, @Private_pct{0}% assigned"),
            ],
        )
    )
    return years_axis(p)

### Preview without a server

The views do not need the app: give them a source and `show()` them, like any Bokeh figure. Click a dot
(tap tool): the same school lights up in both views. Selection lives in the browser, so linked views work
even in a static HTML file. What does need Python is *changing the data*, which is the job of the app.

In [ ]:
latest_p3 = places.query("course == 'P3' and school_year == @LATEST and offered > 0")
preview = ColumnDataSource(school_data(latest_p3))
show(row(make_scatter(preview), make_map(preview), sizing_mode="stretch_width"))

## 6. The app

`schools_app(doc)` is what Bokeh runs for every browser tab that opens the dashboard. It follows the
pattern of any Bokeh server app:

1. **widgets**: one row of filters above everything they control;
2. **sources and views**: empty for now;
3. **callbacks**: `update()` re-filters `places` and replaces the data of every source; `show_school()`
   fills the history chart for one school. Every widget ends up calling one of these two;
4. **layout**, a first `update()`, and `doc.add_root()`.

The comarca → municipality → district dropdowns cascade: choosing a comarca narrows the municipalities and
resets the municipality to *All*; the district dropdown is enabled only for the five cities split into
districts (Barcelona, L'Hospitalet, Sabadell, Terrassa, Santa Coloma).

In [ ]:
COMARQUES = sorted(places["comarca"].unique())
MUNICIPALITIES = sorted(places["municipality"].unique())
# Search box: the latest label of each school, and back from label to school code
LABEL_OF = places.sort_values("year").groupby("school_id")["label"].last()
SCHOOL_OF = pd.Series(LABEL_OF.index, index=LABEL_OF.to_numpy())
NO_SCHOOL = "Click a school (or search for one) to see its history"
SUMMARY_CSS = """
table { border-collapse: collapse; font-size: 13px; }
th, td { padding: 3px 14px; text-align: right; border-bottom: 1px solid #e1e0d9; }
th.row_heading { text-align: left; }
"""


def area_name(comarca, municipality, district):
    """The smallest area selected, for titles."""
    for value in (district, municipality, comarca):
        if value != ALL:
            return value
    return "Catalonia"


def schools_app(doc):
    # 1. Widgets
    course = Select(title="Course", options=COURSES, value="P3", width=110, name="course")
    year = Select(title="School year", options=SCHOOL_YEARS, value=LATEST, width=120, name="year")
    comarca = Select(
        title="Comarca", options=[ALL, *COMARQUES], value=ALL, width=190, name="comarca"
    )
    municipality = Select(
        title="Municipality",
        options=[ALL, *MUNICIPALITIES],
        value=ALL,
        width=190,
        name="municipality",
    )
    district = Select(
        title="District", options=[ALL], value=ALL, width=190, disabled=True, name="district"
    )
    search = AutocompleteInput(
        title="Find a school",
        completions=list(SCHOOL_OF.index),
        width=330,
        search_strategy="includes",
        case_sensitive=False,
        min_characters=3,
        placeholder="3+ letters of its name or code",
        name="search",
    )
    reset = Button(label="Reset", width=90, align="end", name="reset")
    summary = Div(stylesheets=[SUMMARY_CSS], name="summary")

    # 2. Sources and views
    schools = ColumnDataSource(school_data(places.iloc[:0]), name="schools")
    history = ColumnDataSource(history_data(places.iloc[:0]), name="history")
    trend = ColumnDataSource(trend_data(places.iloc[:0]), name="trend")
    scatter, map_, table = make_scatter(schools), make_map(schools), make_table(schools)
    history_fig, history_band = make_history(history)
    trend_fig, trend_band = make_trend(trend)
    history_fig.name = "history_fig"
    selected = {"school_id": None}

    # 3. Callbacks
    def show_school(school_id):
        """Fill the history chart with one school and highlight it in the scatter, map and table."""
        selected["school_id"] = school_id
        rows = places[(places["school_id"] == school_id) & (places["course"] == course.value)]
        history.data = history_data(rows)
        if school_id is None:
            history_fig.title.text = NO_SCHOOL
        elif rows.empty:
            history_fig.title.text = f"{LABEL_OF[school_id]}: never offered {course.value} places"
        else:
            history_fig.title.text = f"{LABEL_OF[school_id]} · {course.value}"
        schools.selected.indices = np.flatnonzero(schools.data["school_id"] == school_id).tolist()

    def update():
        """Re-filter the data for the current widgets and push it to every source."""
        area = {
            "comarca": comarca.value,
            "municipality": municipality.value,
            "district": district.value,
        }
        in_area = filter_places(places, course.value, **area)
        this_year = in_area[in_area["school_year"] == year.value]
        schools.data = school_data(this_year[this_year["offered"] > 0])
        trend.data = trend_data(in_area)

        y, name = int(year.value[:4]), area_name(**area)
        caption = f"<b>{course.value} · {YEAR_LABELS[y]} · {name}</b>"
        summary.text = caption + summary_table(this_year).to_html()
        trend_fig.title.text = f"{course.value} places offered in {name}, by ownership"
        for band in (history_band, trend_band):
            band.left, band.right = y - 0.5, y + 0.5
        show_school(selected["school_id"])

    def on_comarca(attr, old, new):
        in_comarca = places if new == ALL else places[places["comarca"] == new]
        municipality.options = [ALL, *sorted(in_comarca["municipality"].unique())]
        if municipality.value == ALL:
            update()
        else:
            municipality.value = ALL  # its own callback calls update()

    def on_municipality(attr, old, new):
        districts = sorted(places.loc[places["municipality"] == new, "district"].dropna().unique())
        district.options, district.disabled = [ALL, *districts], not districts
        if district.value == ALL:
            update()
        else:
            district.value = ALL  # its own callback calls update()

    def on_select(attr, old, new):
        if new:  # a click on empty space keeps the current school
            show_school(schools.data["school_id"][new[0]])

    def on_search(attr, old, new):
        if new in SCHOOL_OF:
            show_school(SCHOOL_OF[new])

    def on_reset():
        selected["school_id"] = None
        search.value = ""
        course.value, year.value, comarca.value, municipality.value = "P3", LATEST, ALL, ALL
        update()

    for widget in (course, year, district):
        widget.on_change("value", lambda attr, old, new: update())
    comarca.on_change("value", on_comarca)
    municipality.on_change("value", on_municipality)
    search.on_change("value", on_search)
    schools.selected.on_change("indices", on_select)
    reset.on_click(on_reset)

    # 4. Layout
    title = "<h2 style='margin:0'>School places in Catalonia</h2>"
    header = Div(text=f"{title}Pre-enrolment allocation · {DATA_NOTE}")
    doc.add_root(
        column(
            header,
            row(course, year, comarca, municipality, district),
            row(search, reset),
            summary,
            row(scatter, map_, sizing_mode="stretch_width"),
            row(history_fig, trend_fig, sizing_mode="stretch_width"),
            table,
            sizing_mode="stretch_width",
        )
    )
    doc.title = "School places in Catalonia"
    update()

## 7. Test it without a browser

A Bokeh app is Python objects, so we can build it in an empty `Document`, change widgets from code
(each change runs the same callbacks as a click in the browser) and check the sources. This is how the
course CI checks that the dashboard still works.

In [ ]:
from bokeh.document import Document

doc = Document()
schools_app(doc)


def get(name):
    """The widget or source with this name (we named them in schools_app)."""
    return doc.select_one({"name": name})


n_catalonia = len(get("schools").data["school_id"])

get("comarca").value = "Barcelonès"  # cascades to the municipality dropdown
assert "Barcelona" in get("municipality").options and "Girona" not in get("municipality").options
get("municipality").value = "Barcelona"  # enables the district dropdown
assert not get("district").disabled and "Eixample" in get("district").options
n_barcelona = len(get("schools").data["school_id"])
assert 0 < n_barcelona < n_catalonia

get("schools").selected.indices = [0]  # what a click on the first dot does
assert len(get("history").data["year"]) > 0, "the history chart should show the clicked school"

print(f"P3 {LATEST}: {n_catalonia:,} schools in Catalonia, {n_barcelona:,} in Barcelona")
print("History:", get("history_fig").title.text)

## 8. Run it

`show(app)` starts a Bokeh server inside this kernel and embeds its page below. Each widget change makes a
round trip: browser → Python callback → `update()` → new `source.data` → browser. That is why the dashboard
needs a live kernel: exported to a static HTML file (as in notebook 07), the filters would do nothing.

If the output stays blank:
- make `NOTEBOOK_URL` (setup cell) match the address in your browser, e.g. `localhost:8889`;
- open the notebook in JupyterLab, not in VS Code or a GitHub preview;
- after editing a function, re-run its cell and then this one.

To run it as a standalone web app instead, copy the code into `app.py`, end the file with
`schools_app(curdoc())` (from `bokeh.io import curdoc`) and run `bokeh serve --show app.py`.

In [ ]:
show(schools_app, notebook_url=NOTEBOOK_URL)

## 9. Ideas to take it further

- **Colour by a number.** Colour the map by `occupancy_pct` with a sequential palette and a `ColorBar`
  (notebook 07 shows how). Which areas run out of P3 places?
- **Fresh data.** When the next school year is published, set `SOURCE = "api"` (or refresh the snapshot
  with `scripts/download_data.py`). Does the drop in P3 places continue?
- **Hide a group.** Draw public and private schools as two renderers, each with
  `CDSView(filter=GroupFilter(column_name="ownership", group=...))`, and set `legend.click_policy = "hide"`.
- **Two moments.** For each school, compare the first and the last school year in a dumbbell chart: which
  schools lost the most first-choice demand?
- **No server.** Rewrite the course filter as a `CustomJS` callback so the page works as a static HTML file.
  Where does that approach stop being practical?